<a href="https://colab.research.google.com/github/veridelisi/The-First-Bank-of-United-States-1791-1793-/blob/main/Employment%2C_unemployment%2C_and_participation_rates_by_sex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
from io import StringIO

# ============================================================
# 1. DOWNLOAD FEMALE AND TOTAL FOREIGN-BORN STOCK
# ============================================================

url = (
    "https://sdmx.oecd.org/public/rest/data/"
    "OECD.ELS.IMD,DSD_MIG_F@DF_MIG_POPF,1.0/"
    ".W.A.._T+F...?"
    "startPeriod=2015&"
    "endPeriod=2024&"
    "dimensionAtObservation=AllDimensions"
)

response = requests.get(
    url,
    headers={"Accept": "text/csv"},
    timeout=60
)

response.raise_for_status()

df_stock = pd.read_csv(StringIO(response.text))

# Convert values to numeric
df_stock["TIME_PERIOD"] = pd.to_numeric(
    df_stock["TIME_PERIOD"],
    errors="coerce"
)

df_stock["OBS_VALUE"] = pd.to_numeric(
    df_stock["OBS_VALUE"],
    errors="coerce"
)

df_stock = df_stock.dropna(
    subset=["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]
).copy()

df_stock["TIME_PERIOD"] = df_stock["TIME_PERIOD"].astype(int)

print("SEX codes returned by API:")
print(df_stock["SEX"].unique())

# ============================================================
# 2. STANDARDISE SEX CODES
# ============================================================

df_stock["SEX"] = (
    df_stock["SEX"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        "T": "_T",
        "TOTAL": "_T",
        "FEMALE": "F"
    })
)

# Keep Female and Total
selected = df_stock[
    df_stock["SEX"].isin(["F", "_T"])
].copy()

# ============================================================
# 3. PLACE FEMALE AND TOTAL VALUES SIDE BY SIDE
# ============================================================

stock_wide = (
    selected
    .pivot_table(
        index=["REF_AREA", "TIME_PERIOD"],
        columns="SEX",
        values="OBS_VALUE",
        aggfunc="first"
    )
    .reset_index()
)

stock_wide.columns.name = None

print("Columns after pivot:")
print(stock_wide.columns.tolist())

if "F" not in stock_wide.columns or "_T" not in stock_wide.columns:
    raise ValueError(
        "Female and Total observations were not both returned. "
        f"Available columns: {stock_wide.columns.tolist()}"
    )

stock_wide = stock_wide.rename(
    columns={
        "F": "Female_Stock",
        "_T": "Total_Stock"
    }
)

# Missing values are excluded
stock_wide = stock_wide.dropna(
    subset=["Female_Stock", "Total_Stock"]
).copy()

# Exclude zero or invalid totals
stock_wide = stock_wide[
    stock_wide["Total_Stock"] > 0
].copy()

# ============================================================
# 4. CALCULATE MALE STOCK AND SHARES
# ============================================================

stock_wide["Male_Stock"] = (
    stock_wide["Total_Stock"] -
    stock_wide["Female_Stock"]
)

# Remove logically invalid observations
stock_wide = stock_wide[
    (stock_wide["Female_Stock"] >= 0) &
    (stock_wide["Male_Stock"] >= 0)
].copy()

stock_wide["Female_Share"] = (
    stock_wide["Female_Stock"] /
    stock_wide["Total_Stock"] * 100
)

stock_wide["Male_Share"] = (
    stock_wide["Male_Stock"] /
    stock_wide["Total_Stock"] * 100
)

# ============================================================
# 5. FINAL COUNTRY-YEAR DATA
# ============================================================

share_data = (
    stock_wide[
        [
            "REF_AREA",
            "TIME_PERIOD",
            "Female_Stock",
            "Male_Stock",
            "Total_Stock",
            "Female_Share",
            "Male_Share"
        ]
    ]
    .sort_values(["REF_AREA", "TIME_PERIOD"])
    .reset_index(drop=True)
)

share_data[
    ["Female_Share", "Male_Share"]
] = share_data[
    ["Female_Share", "Male_Share"]
].round(2)

display(share_data)

# Check: shares must add up to 100
share_data["Share_Check"] = (
    share_data["Female_Share"] +
    share_data["Male_Share"]
).round(2)

print("\nShare check:")
print(share_data["Share_Check"].value_counts().head())

SEX codes returned by API:
['F' '_T']
Columns after pivot:
['REF_AREA', 'TIME_PERIOD', 'F', '_T']


,REF_AREA,TIME_PERIOD,Female_Stock,Male_Stock,Total_Stock,Female_Share,Male_Share
0,AUS,2015,3423870.0,3305860.0,6729730.0,50.88,49.12
1,AUS,2016,3524620.0,3387490.0,6912110.0,50.99,49.01
2,AUS,2017,3646390.0,3495640.0,7142030.0,51.06,48.94
3,AUS,2018,3754140.0,3592380.0,7346520.0,51.10,48.90
4,AUS,2019,3860790.0,3688870.0,7549660.0,51.14,48.86
...,...,...,...,...,...,...,...
256,TUR,2024,1569608.0,1361213.0,2930821.0,53.56,46.44
257,USA,2020,22879690.0,21378560.0,44258250.0,51.70,48.30
258,USA,2021,23341810.0,21931135.0,45272945.0,51.56,48.44
259,USA,2022,24225290.0,23106190.0,47331480.0,51.18,48.82



Share check:
Share_Check
100.0    261
Name: count, dtype: int64
